# Rainfall Analysis and Planting Advisory
Cleans monthly rainfall data for five Kenyan counties, analyses it with NumPy and produces a planting advisory for a farmer-support NGO.

In [1]:
import csv
import numpy as np

months = np.array(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])

counties = ["Nairobi", "Nakuru", "Kisumu", "Mombasa", "Machakos"]

clean_rows = [
    [64, 56, 92, 219, 176, 35, 18, 24, 31, 61, 165, 104],   # Nairobi
    [35, 42, 70, 145, 120, 78, 96, 105, 72, 55, 60, 38],    # Nakuru
    [58, 90, 150, 240, 180, 95, 70, 85, 90, 88, 130, 95],   # Kisumu
    [30, 15, 55, 150, 240, 110, 90, 65, 60, 90, 100, 70],   # Mombasa
]

machakos_raw = ["48", "35", "88", "N/A", "140", " 20 ",
                "9", "12", "", "45", "182", "96"]

## 1. Clean the Machakos data
Each value is converted with `int()`. `int("N/A")` and `int("")` raise `ValueError`, which is caught specifically. `int(" 20 ")` works because `int()` ignores surrounding whitespace. Invalid months are filled with the rounded average of the valid values.

In [2]:
def clean_machakos(raw):
    """Convert raw strings to ints; impute invalid entries with the rounded mean of valid ones."""
    values, bad_idx = [], []
    for i, item in enumerate(raw):
        try:
            values.append(int(item))
        except ValueError:
            values.append(None)
            bad_idx.append(i)
    valid = [v for v in values if v is not None]
    fill = round(sum(valid) / len(valid))
    cleaned = [fill if v is None else v for v in values]
    return cleaned, bad_idx, fill

machakos_clean, imputed_idx, fill_value = clean_machakos(machakos_raw)
print("Imputed months:", [str(months[i]) for i in imputed_idx])
print("Imputed value (rounded mean of valid readings):", fill_value, "mm")
print("Cleaned Machakos row:", machakos_clean)

Imputed months: ['Apr', 'Sep']
Imputed value (rounded mean of valid readings): 68 mm
Cleaned Machakos row: [48, 35, 88, 68, 140, 20, 9, 12, 68, 45, 182, 96]


## 2. Combine into a 5 × 12 NumPy array

In [3]:
rain = np.array(clean_rows + [machakos_clean])
print("Shape:", rain.shape)
print("dtype:", rain.dtype)

Shape: (5, 12)
dtype: int64


## 3. Annual totals and monthly averages

In [4]:
annual_totals = rain.sum(axis=1)
monthly_avg = rain.mean(axis=0)

for county, total in zip(counties, annual_totals):
    print(f"{county:<9} annual total: {total} mm")

print()
for month, avg in zip(months, monthly_avg):
    print(f"{month}: {avg:.1f} mm")

Nairobi   annual total: 1045 mm
Nakuru    annual total: 916 mm
Kisumu    annual total: 1371 mm
Mombasa   annual total: 1075 mm
Machakos  annual total: 811 mm

Jan: 47.0 mm
Feb: 47.6 mm
Mar: 91.0 mm
Apr: 164.4 mm
May: 171.2 mm
Jun: 67.6 mm
Jul: 56.6 mm
Aug: 58.2 mm
Sep: 64.2 mm
Oct: 67.8 mm
Nov: 127.4 mm
Dec: 80.6 mm


## 4. Wettest month per county

In [5]:
wettest_idx = np.argmax(rain, axis=1)
wettest_month = months[wettest_idx]
for county, month, mm in zip(counties, wettest_month, rain.max(axis=1)):
    print(f"{county:<9} wettest month: {month} ({mm} mm)")

Nairobi   wettest month: Apr (219 mm)
Nakuru    wettest month: Apr (145 mm)
Kisumu    wettest month: Apr (240 mm)
Mombasa   wettest month: May (240 mm)
Machakos  wettest month: Nov (182 mm)


## 5. Dry months (below 50 mm)

In [6]:
dry_mask = rain < 50
dry_months = dry_mask.sum(axis=1)
for county, n in zip(counties, dry_months):
    print(f"{county:<9} dry months: {n}")

Nairobi   dry months: 4
Nakuru    dry months: 3
Kisumu    dry months: 0
Mombasa   dry months: 2
Machakos  dry months: 6


## 6. Planting window (two consecutive months with highest combined rainfall)
`rain[:, :-1] + rain[:, 1:]` adds each month to the next, so column *j* holds the combined rainfall of month *j* and month *j+1*.

In [7]:
pair_sums = rain[:, :-1] + rain[:, 1:]
start_idx = np.argmax(pair_sums, axis=1)
planting_window = np.char.add(np.char.add(months[start_idx], "-"), months[start_idx + 1])
best_totals = pair_sums.max(axis=1)
for county, window, total in zip(counties, planting_window, best_totals):
    print(f"{county:<9} planting window: {window} ({total} mm combined)")

Nairobi   planting window: Apr-May (395 mm combined)
Nakuru    planting window: Apr-May (265 mm combined)
Kisumu    planting window: Apr-May (420 mm combined)
Mombasa   planting window: Apr-May (390 mm combined)
Machakos  planting window: Nov-Dec (278 mm combined)


## 7. Write rainfall_summary.csv

In [8]:
with open("rainfall_summary.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["county", "annual_total_mm", "wettest_month", "dry_months", "planting_window"])
    for row in zip(counties, annual_totals.tolist(), wettest_month.tolist(),
                   dry_months.tolist(), planting_window.tolist()):
        writer.writerow(row)

with open("rainfall_summary.csv") as f:
    print(f.read())

county,annual_total_mm,wettest_month,dry_months,planting_window
Nairobi,1045,Apr,4,Apr-May
Nakuru,916,Apr,3,Apr-May
Kisumu,1371,Apr,0,Apr-May
Mombasa,1075,May,2,Apr-May
Machakos,811,Nov,6,Nov-Dec



## Planting Advisory
For Nairobi, Nakuru, Kisumu and Mombasa, the best planting window is April to May, when the long rains deliver the highest two-month rainfall (390 mm in Mombasa, up to 420 mm in Kisumu). Machakos is different: its strongest window is November to December (278 mm), so farmers there should plan around the short rains rather than the April-May season. Kisumu is the most reliably wet county (1,371 mm a year, no dry months), while Machakos is the driest, with 6 months below 50 mm, so drought-tolerant varieties are advisable there. Nakuru and Nairobi have 3 and 4 dry months respectively, so farmers should avoid planting before the rains are properly established. Please note that one Machakos station's data was typed by hand and two months (April and September) were missing, so we estimated them with the average of the other months. Machakos totals and its dry-month count may therefore be slightly off, and the advice should be confirmed with local extension officers.

## Data Quality Notes
- The Machakos row arrived as raw text. Two entries, April (`"N/A"`) and September (`""`), could not be converted to numbers and raised `ValueError`.
- Both were replaced with the rounded mean of the 10 valid Machakos readings (68 mm). This is an estimate, not a measurement, and it smooths out any real rainfall peaks or dips in those months.
- `" 20 "` (June) contained extra spaces but was a valid number and was converted normally.
- Imputation does not change the Machakos wettest month (November) or planting window (Nov-Dec), but it does affect the annual total (811 mm).
- Each county is represented by one monthly series, with no information on the year or station coverage, so results indicate typical patterns, not guarantees.